# 0.94644 + a from-scratch LightGBM: what a new strong member does to a saturated blend

The public top blend is 0.94644 and, as I measured over two days and sixteen
submissions, every public source you add to it is flat or negative
([here](https://www.kaggle.com/code/megayak/s6e9-lb-0-94643-and-six-missing-sources)).
The reason was always the same: the candidates that were decorrelated were too
weak, and the strong ones were too correlated.

So I built a member that is both — a single LightGBM from raw data at
[CV 0.9463 / LB 0.94633](https://www.kaggle.com/code/megayak/s6e9-one-lightgbm-from-raw-data-cv-0-9463),
Spearman 0.9978 against the blend's strongest component — and added it. This
notebook is that blend, with the sweep.

| addition to the 0.94644 blend | public LB |
|---|---|
| none | 0.94644 |
| + 10% hybrid LGBM | 0.94644 |
| + 20% hybrid LGBM | 0.94644 |
| + 20% hybrid LGBM, 3-seed average | 0.94644 |
| + 30% hybrid LGBM | 0.94643 |
| *hybrid + najiama V3 only, 50/50* | 0.94638 |
| *hybrid alone* | 0.94633 |

Flat again. A member that is stronger and more decorrelated than anything the
blend already contained did not move it. **The blend is not short of members. It
is at the resolution limit of what rank-averaging these sources can express.**
Going above 0.94644 needs a better *base model*, not a better mix — which is the
direction the single-model notebook is for.

In [ ]:
import glob
import numpy as np
import pandas as pd
from scipy.stats import rankdata, spearmanr

def find_one(*pats):
    for p in pats:
        h = sorted(glob.glob(f"/kaggle/input/**/{p}", recursive=True))
        if h:
            return h[0]
    raise FileNotFoundError(" | ".join(pats))

ID, TARGET = "id", "Will_Buy_EV"
rank01 = lambda s: rankdata(s) / len(s)

def load(*pats):
    p = find_one(*pats); print("  ", p)
    return pd.read_csv(p).sort_values(ID)[TARGET].to_numpy()

test_id = pd.read_csv(find_one("test.csv"), usecols=[ID]).sort_values(ID)[ID].to_numpy()
print("sources:")
nina     = load("ps-s6e9-h-blend-1/submission.csv", "nina2025/**/submission.csv")
zoomzoom = load("submission_curvature9_lgbpair05.csv")
kospintr = load("evehicle-stacked-lgbm-catb-xgb-hgbc-baseline/submission.csv", "kospintr/**/submission.csv")
mikhail  = load("electric-vehicle-purchases-single-xgb/submission.csv",
                "electric-vehicle-purchases-xgb/submission.csv", "mikhailnaumov/**/submission.csv")
najiama  = load("pure-lgbm-model-cv-0-94606-lb-0-94637/submission_LIGHTGBM.csv", "**/submission_LIGHTGBM.csv")
realmlp  = load("ps-s6-e9-realmlp-pytorch/submission.csv", "yekenot/**/submission.csv")
hybrid   = load("s6e9-one-lightgbm-from-raw-data-cv-0-9463/submission.csv", "megayak/**/submission.csv")

# talhatursun's daily recipe, unchanged
base  = 0.5 * rank01(nina) + 0.5 * rank01(zoomzoom)
E     = 0.90 * base + 0.05 * rank01(kospintr) + 0.05 * rank01(mikhail)
F     = 0.50 * rank01(E) + 0.50 * rank01(najiama)
daily = rank01(0.85 * rank01(F) + 0.15 * rank01(realmlp))

# the new member
W = 0.20
final = rank01((1 - W) * daily + W * rank01(hybrid))

print(f"\nhybrid vs daily blend  spearman = {spearmanr(rank01(hybrid), daily).statistic:.5f}")
print(f"hybrid vs najiama V3   spearman = {spearmanr(rank01(hybrid), rank01(najiama)).statistic:.5f}")
pd.DataFrame({ID: test_id, TARGET: final}).to_csv("submission.csv", index=False)
print(f"wrote submission.csv  ({1-W:.0%} daily + {W:.0%} hybrid)")

## Why a good member still does nothing

Three of the six sources in the daily blend are themselves blends of ten-plus
models. The rank-average of that many correlated rankings has already converged
on a consensus ordering; a seventh vote, even a well-informed one at 20%, changes
the ordering of only a handful of pairs, and at 286,571 test rows a handful of
pairs is below the fifth decimal.

The two-model result is the tell: hybrid + V3 at 50/50 scores 0.94638, well above
either alone (0.94633, 0.94637). Two strong models *do* combine. Add the same
hybrid to a blend that already holds V3 plus five others and the gain is gone —
the blend had already extracted what V3 knows, and the hybrid mostly agrees
with V3 (0.9978).

**The lever left is the base.** A single model at 0.9465 would move this blend.
Nothing at 0.9463 will. That is why the
[single-model notebook](https://www.kaggle.com/code/megayak/s6e9-one-lightgbm-from-raw-data-cv-0-9463)
publishes its OOF: the next gain here comes from someone training a stronger
model, not from anyone re-weighting these six.

---

Credit: recipe and sources as in
[@talhatursun's daily blend](https://www.kaggle.com/code/talhatursun/s6e9-daily-rank-average-ensemble);
members [@nina2025](https://www.kaggle.com/code/nina2025/ps-s6e9-h-blend-1),
[@jazivxt](https://www.kaggle.com/datasets/jazivxt/s6e9-zoom-zoom-baseline),
[@kospintr](https://www.kaggle.com/code/kospintr/evehicle-stacked-lgbm-catb-xgb-hgbc-baseline),
[@mikhailnaumov](https://www.kaggle.com/code/mikhailnaumov/electric-vehicle-purchases-single-xgb),
[@najiama](https://www.kaggle.com/code/najiama/pure-lgbm-model-cv-0-94606-lb-0-94637),
[@yekenot](https://www.kaggle.com/code/yekenot/ps-s6-e9-realmlp-pytorch).
The hybrid model and the measurements are mine. If this saved you submissions,
an upvote helps.